# Snell's law in layered media

*optics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/snell_refraction)

A fan of light rays crosses three layers with different refractive indices. The vector form of Snell's law bends every ray at once with one einsum, and switches to a mirror reflection wherever total internal reflection kicks in.

**Vector form**

$$\mathbf t = \eta\,\mathbf d + \left(\eta\cos\theta_i - \sqrt{1-\eta^2(1-\cos^2\theta_i)}\right)\mathbf n,\quad \eta = \tfrac{n_1}{n_2},\ \cos\theta_i = -\mathbf n\cdot\mathbf d$$

**Index (tensor) form**

$$t_{ri} = \eta_r\,d_{ri} + \left(\eta_r c_r - \sqrt{1-\eta_r^2(1-c_r^2)}\right) n_{ri},\quad c_r = -n_{rj}\,d_{rj}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $d_{ri}$ | `d` | (R, 2) | unit direction of ray $r$, component $i \in \{x, y\}$ |
| $n_{ri}$ | `normal` | (R, 2) | unit surface normal facing the incoming ray |
| $\eta_r$ | `n_layer[layer] / n_layer[next_layer]` | (R,) | index ratio $n_1/n_2$ at the interface each ray hits |
| $c_r = -n_{rj}d_{rj}$ | `-einsum('rj,rj->r', n, d)` | (R,) | repeated $j$ ⇒ summed: $\cos\theta_i$ of each ray |
| $t_{ri}$ | `refract(d, n, eta)` | (R, 2) | refracted direction (still a unit vector) |
| $r_{ri} = d_{ri} - 2(d_{rj}n_{rj})n_{ri}$ | `reflect(d, n)` | (R, 2) | mirror direction, used where $1 - \eta^2(1 - c^2) < 0$ (TIR) |

Rays travel at speed $c/n$, so the dashed line through the ray tips is a wavefront: it lags behind in dense layers. With the defaults the bottom layer is air ($n = 1$); rays steeper than the critical angle $\arcsin(n_3/n_2)$ at that interface are totally internally reflected and head back up. Set two indices equal to make an interface vanish.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/snell_refraction/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_layer = tensors.indices(params["n1"], params["n2"], params["n3"], dev)
    angles = tensors.fan_angles(params["n_rays"], params["max_angle"])
    p0, d0 = tensors.initial_rays(angles, dev)

    times, rays = solve.solve(p0, d0, n_layer, params["t_max"])

    for a, b in zip(n_layer[:-1].tolist(), n_layer[1:].tolist()):
        theta_c = solve.critical_angle(a, b)
        note = f"critical angle {theta_c:.1f}°" if theta_c else "no total internal reflection"
        print(f"interface n = {a:.2f} -> {b:.2f}: {note}")
    return plot.animate(times, rays, n_layer, angles, out_dir / "snell_refraction.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/snell_refraction/equation.py
r"""Physics → tensor translation.

Snell's law  n_1 \sin\theta_1 = n_2 \sin\theta_2  in vector form, for a unit direction
\mathbf d hitting a surface with unit normal \mathbf n (pointing back towards the incoming ray):

    \mathbf t = \eta\,\mathbf d + (\eta\cos\theta_i - \cos\theta_t)\,\mathbf n,
    \eta = n_1/n_2,  \cos\theta_i = -\mathbf n\cdot\mathbf d,
    \cos^2\theta_t = 1 - \eta^2 (1 - \cos^2\theta_i)

In index notation for a batch of rays r and spatial components i:

    c_r = -n_{rj} d_{rj}                       (repeated j ⇒ summed: a dot product per ray)
    t_{ri} = \eta_r d_{ri} + (\eta_r c_r - \sqrt{1 - \eta_r^2 (1 - c_r^2)})\, n_{ri}

When the square root's argument is negative there is no refracted ray: total internal
reflection, and the ray is mirrored instead,

    r_{ri} = d_{ri} - 2 (d_{rj} n_{rj})\, n_{ri}
"""

import torch


def cos_incidence(d: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""c_r = -n_{rj} d_{rj}.   d, n: (R, 2) → (R,)"""
    return -torch.einsum("rj,rj->r", n, d)


def reflect(d: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""r_{ri} = d_{ri} - 2 (d_{rj} n_{rj}) n_{ri}.   (R, 2) → (R, 2)"""
    d_dot_n = torch.einsum("rj,rj->r", d, n)  # (R,)
    return d - 2 * torch.einsum("r,ri->ri", d_dot_n, n)


def refract(d: torch.Tensor, n: torch.Tensor, eta: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Vector Snell's law for every ray at once.

    d: (R, 2) unit directions, n: (R, 2) unit normals facing the incoming ray, eta: (R,) = n_1/n_2.
    Returns (new directions (R, 2), tir (R,) bool mask of totally internally reflected rays).
    """
    cos_i = cos_incidence(d, n)  # (R,)
    cos_t_sq = 1 - eta**2 * (1 - cos_i**2)  # (R,)   < 0  ⇔  total internal reflection
    tir = cos_t_sq < 0
    cos_t = torch.sqrt(cos_t_sq.clamp(min=0))  # (R,)
    t = torch.einsum("r,ri->ri", eta, d) + torch.einsum("r,ri->ri", eta * cos_i - cos_t, n)  # (R, 2)
    return torch.where(tir[:, None], reflect(d, n), t), tir


def sin_angle(d: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""|\sin\theta| between direction and normal = |2D cross product| = |d_x n_y - d_y n_x|.   (R, 2) → (R,)"""
    return (d[:, 0] * n[:, 1] - d[:, 1] * n[:, 0]).abs()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/snell_refraction/tensors.py
"""Build the concrete tensors: the stack of layers and the fan of rays leaving the source."""

import torch

THICKNESS = 1.0  # every layer is 1 unit thick: layer k fills  -(k+1) < y < -k
WIDTH = 5.0  # the box spans 0 < x < WIDTH


def indices(n1: float, n2: float, n3: float, device: torch.device) -> torch.Tensor:
    """Refractive index of each layer, top to bottom: shape (L,) = (3,)."""
    return torch.tensor([n1, n2, n3], device=device)


def fan_angles(n_rays: int, max_angle_deg: float) -> torch.Tensor:
    """Angles from the vertical, spread from 5° to max_angle, in radians: shape (R,)."""
    return torch.deg2rad(torch.linspace(5.0, max_angle_deg, n_rays))


def initial_rays(angles: torch.Tensor, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Every ray starts at the top-left corner and heads down-right.

    p[r, :] = (0, 0)                        position p_{ri}          (R, 2)
    d[r, :] = (sin θ_r, −cos θ_r)           unit direction d_{ri}    (R, 2)
    """
    angles = angles.to(device)
    d = torch.stack([torch.sin(angles), -torch.cos(angles)], dim=-1)  # (R, 2)
    p = torch.zeros_like(d)
    return p, d


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/snell_refraction/solve.py
"""March every ray forward in time; at each layer boundary apply Snell's law (or TIR)."""

import math

import torch

from .equation import refract
from .tensors import THICKNESS, WIDTH


def solve(
    p: torch.Tensor, d: torch.Tensor, n_layer: torch.Tensor, t_max: float, dt: float = 0.01
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), positions (T, R, 2)).

    A ray moves at speed c/n (c = 1) along d. When a step would carry it out of its layer we stop
    it on the interface and turn its direction with refract(). Rays leaving the box freeze.
    """
    n_layers = len(n_layer)
    n_steps = int(round(t_max / dt))
    layer = torch.zeros(len(p), dtype=torch.long, device=p.device)  # (R,) layer each ray is in
    alive = torch.ones(len(p), dtype=torch.bool, device=p.device)  # (R,)
    path = [p]
    for _ in range(n_steps):
        speed = 1.0 / n_layer[layer]  # (R,)   c / n
        p_new = p + dt * speed[:, None] * d  # (R, 2)

        going_down = d[:, 1] < 0  # (R,)
        y_wall = -torch.where(going_down, layer + 1, layer).float() * THICKNESS  # (R,) boundary ahead
        next_layer = torch.where(going_down, layer + 1, layer - 1)  # (R,)  -1 or L means "outside"
        crossing = alive & torch.where(going_down, p_new[:, 1] < y_wall, p_new[:, 1] > y_wall)
        inner = crossing & (next_layer >= 0) & (next_layer < n_layers)  # a real interface, not the box edge

        if inner.any():
            # stop the ray exactly on the interface
            s = (y_wall - p[:, 1]) / (p_new[:, 1] - p[:, 1])  # (R,) fraction of the step taken
            p_hit = p + s[:, None] * (p_new - p)
            # the normal faces the incoming ray: +y for rays going down, −y for rays going up
            normal = torch.zeros_like(d)
            normal[:, 1] = torch.where(going_down, 1.0, -1.0)
            eta = n_layer[layer] / n_layer[next_layer.clamp(0, n_layers - 1)]  # (R,)   n_1 / n_2
            d_new, tir = refract(d, normal, eta)
            p_new = torch.where(inner[:, None], p_hit, p_new)
            d = torch.where(inner[:, None], d_new, d)
            layer = torch.where(inner & ~tir, next_layer, layer)  # a reflected ray stays in its layer

        p = torch.where(alive[:, None], p_new, p)  # frozen rays stay where they left the box
        alive = alive & ~(crossing & ~inner) & (p[:, 0] < WIDTH)
        path.append(p)

    times = torch.arange(n_steps + 1, device=p.device) * dt
    return times, torch.stack(path)


def critical_angle(n1: float, n2: float) -> float | None:
    """θ_c = arcsin(n2 / n1) in degrees, or None when n1 ≤ n2 (no TIR possible)."""
    return math.degrees(math.asin(n2 / n1)) if n1 > n2 else None


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/snell_refraction/plot.py
"""Animate the rays: tips advance at speed c/n and bend at every interface; a dashed line joins the tips."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .tensors import THICKNESS, WIDTH


def animate(times: torch.Tensor, rays: torch.Tensor, n_layer: torch.Tensor, angles: torch.Tensor, path: Path) -> Path:
    pos = rays.cpu().numpy()  # (T, R, 2)
    t = times.cpu().numpy()
    n = n_layer.cpu().tolist()
    degrees = torch.rad2deg(angles).cpu().numpy()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(0, WIDTH)
    ax.set_ylim(-len(n) * THICKNESS, 0)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(False)
    shade = plt.get_cmap("Blues")
    for k, nk in enumerate(n):  # one band per layer, darker = optically denser
        ax.axhspan(-(k + 1) * THICKNESS, -k * THICKNESS, color=shade(0.08 + 0.4 * (nk - 1)), zorder=0)
        ax.text(WIDTH - 0.08, -(k + 0.5) * THICKNESS, f"n = {nk:.2f}", ha="right", va="center", fontsize=9)

    colors = plt.get_cmap("plasma")(np.linspace(0, 0.85, len(degrees)))
    lines = [ax.plot([], [], lw=1.4, color=c)[0] for c in colors]
    dots = [ax.plot([], [], "o", ms=3, color=c)[0] for c in colors]
    (front,) = ax.plot([], [], "--", lw=0.8, color="#374151")
    clock = ax.text(0.02, 0.04, "", transform=ax.transAxes, fontsize=9)
    ax.set_title(f"{len(degrees)} rays, {degrees[0]:.0f}°–{degrees[-1]:.0f}° from the vertical")

    def update(step: int):
        for r, (line, dot) in enumerate(zip(lines, dots)):
            line.set_data(pos[: step + 1, r, 0], pos[: step + 1, r, 1])
            dot.set_data([pos[step, r, 0]], [pos[step, r, 1]])
        moving = (pos[step] != pos[max(step - 1, 0)]).any(-1)  # (R,) rays still inside the box
        tips = np.where(moving[:, None], pos[step], np.nan)  # NaN hides frozen rays from the line
        front.set_data(tips[:, 0], tips[:, 1])  # tips reached at equal time ≈ a wavefront
        clock.set_text(f"t = {t[step]:.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n1': 1.6,
    'n2': 1.33,
    'n3': 1.0,
    'n_rays': 12,
    'max_angle': 70.0,
    't_max': 8.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.snell_refraction")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.snell_refraction.main")
problem_dir = Path("problems/optics/snell_refraction")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
